# Proyecto Primer Parcial — Aporte individual

**Estudiante:** Marvin Alex Galán Martínez  
**Grupo:** D  
**Paralelo:** CDDEIA-ELNO-4-2  
**Responsabilidad:** Preprocesamiento y Pipeline  
**Dataset:** `student-mat.csv` (395 estudiantes)



## Preparación del entorno y carga de datos

Estas celdas son de apoyo técnico y comunes al grupo, para que el notebook se ejecute de forma independiente, mi aporte individual empieza en **Preprocesamiento y Pipeline**.

In [15]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeCV, LassoCV
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             accuracy_score, precision_score, recall_score, confusion_matrix)
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

SEMILLA = 2026
pd.set_option('display.max_columns', 40)
sns.set_theme(style='whitegrid', context='notebook')
print('Bibliotecas cargadas; semilla:', SEMILLA)

Bibliotecas cargadas; semilla: 2026


In [19]:

rutas = [Path('student-mat.csv'), Path('data/student-mat.csv'), Path('/content/student-mat.csv')]
ruta_csv = next((p for p in rutas if p.is_file()), None)
if ruta_csv is None:
    raise FileNotFoundError('Coloca student-mat.csv junto al notebook o en data/. No se modificó el dataset.')
df_original = pd.read_csv(ruta_csv, sep=',')
assert df_original.shape == (395, 33), f'Se esperaban 395 filas y 33 columnas, recibido: {df_original.shape}'
assert {'G1','G2','G3','absences','studytime'}.issubset(df_original.columns)
print('Archivo:', ruta_csv, '| Filas:', df_original.shape[0], '| Columnas:', df_original.shape[1])
display(df_original.head())

Archivo: student-mat.csv | Filas: 395 | Columnas: 33


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,other,mother,1,2,3,yes,no,yes,no,yes,yes,yes,no,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,home,mother,1,3,0,no,yes,yes,yes,yes,yes,yes,yes,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,home,father,1,2,0,no,yes,yes,no,yes,yes,no,no,4,3,2,1,2,5,4,6,10,10


## Preprocesamiento y Pipeline

**Revisión personal:** Comprobaciòn de  nulos inducidos del 5 %, valores IQR y ColumnTransformer; justificaciòn de mediana, moda y escalado.

## Parte 2 — Preprocesamiento y prevención de fuga de datos


El conjunto original no tiene faltantes. Introducimos aleatoriamente un **5 % (20 de 395)** en `absences` y otro **5 % (20 de 395)** en `studytime`. No modificamos las notas ni las variables objetivo. Las ausencias pueden tener valores altos legítimos; se conservarán después de identificarlos mediante IQR. La imputación usa mediana en numéricas, moda en categóricas; las numéricas se estandarizan y las categóricas se codifican con One-Hot. Los parámetros se aprenden solo en entrenamiento.

In [17]:
datos = df_original.copy(deep=True)
rng = np.random.default_rng(SEMILLA)
for variable in ['absences','studytime']:
    posiciones = rng.choice(datos.index.to_numpy(), size=round(0.05 * len(datos)), replace=False)
    datos.loc[posiciones, variable] = np.nan
print('Nulos después de introducir el 5 % de forma reproducible:')
display(datos[['absences','studytime']].isna().sum().to_frame('nulos'))
q1, q3 = df_original['absences'].quantile([0.25, 0.75]); iqr = q3-q1
lim_inf, lim_sup = q1 - 1.5*iqr, q3 + 1.5*iqr
atipicos = df_original['absences'].lt(lim_inf) | df_original['absences'].gt(lim_sup)
print(f'Outliers IQR en absences: {atipicos.sum()} | Límites: {lim_inf:.1f} y {lim_sup:.1f}')
print('Decisión: conservarlos. Son ausencias posibles; eliminarlos sin evidencia podría sesgar el análisis.')

Nulos después de introducir el 5 % de forma reproducible:


,nulos
absences,20
studytime,20


Outliers IQR en absences: 15 | Límites: -12.0 y 20.0
Decisión: conservarlos. Son ausencias posibles; eliminarlos sin evidencia podría sesgar el análisis.


In [18]:
columnas_A = [c for c in datos.columns if c != 'G3']
columnas_B = [c for c in datos.columns if c not in {'G1','G2','G3'}]
objetivo_reg = datos['G3']
objetivo_aprueba = (datos['G3'] >= 10).astype(int)
idx_entrenamiento, idx_prueba = train_test_split(
    np.arange(len(datos)), test_size=0.20, random_state=SEMILLA, stratify=objetivo_aprueba
)
print('Entrenamiento:', len(idx_entrenamiento), '| Prueba:', len(idx_prueba))

def construir_preprocesador(columnas: list[str]) -> ColumnTransformer:
    explicativas = datos[columnas]
    numericas = explicativas.select_dtypes(include=np.number).columns.tolist()
    categoricas = explicativas.select_dtypes(exclude=np.number).columns.tolist()
    return ColumnTransformer(transformers=[
        ('numericas', Pipeline(steps=[
            ('imputar', SimpleImputer(strategy='median')),
            ('escalar', StandardScaler())
        ]), numericas),
        ('categoricas', Pipeline(steps=[
            ('imputar', SimpleImputer(strategy='most_frequent')),
            ('codificar', OneHotEncoder(handle_unknown='ignore', drop='first'))
        ]), categoricas)
    ], remainder='drop')

print('Modelo A:', len(columnas_A), 'variables originales; Modelo B:', len(columnas_B))

Entrenamiento: 316 | Prueba: 79
Modelo A: 32 variables originales; Modelo B: 30


## Registro de aporte personal


- **Qué revisé o cambié:** Revisé la parte de preprocesamiento y pipeline. Comprobé que los nulos inducidos fueran exactamente el 5 % (20 de 395) en `absences` y en `studytime`, y que fueran reproducibles con la semilla 2026. Revisé los outliers de `absences` con el método IQR y la decisión de conservarlos. Revisé la división entrenamiento/prueba (80/20, estratificada por aprobación) y el `ColumnTransformer`: mediana + escalado en numéricas, moda + One-Hot en categóricas.
- **Qué aprendí:** Aprendí que imputar y escalar debe hacerse dentro de un pipeline para que los parámetros (mediana, moda, media, desviación) se calculen solo con los datos de entrenamiento. Si se calculan con todos los datos, la información de la prueba se filtra al entrenamiento (fuga de datos).Usé la mediana porque no se ve afectada por esos valores extremos, y la moda para las categóricas porque no tienen promedio.

- **Qué resultado comprobé:** La ejecución dio 20 nulos en `absences` y 20 en `studytime`. Con el IQR, los límites fueron -12.0 y 20.0, y se detectaron 15 outliers en `absences`. La división dejó 316 estudiantes en entrenamiento y 79 en prueba. El Modelo A usa 32 variables y el Modelo B usa 30, porque excluye G1 y G2.

### Uso de herramientas
Se utilizaron herramientas de inteligencia artificial generativa (ChatGPT y Claude) como apoyo técnico para estructurar el código y correcciòn del anàlisis.